In [2]:
import torch
ckpt_path = '/home/obed/scratch/dino_pca/4490486/dino_256_0/best.pth'

state = torch.load(ckpt_path, map_location='cpu', weights_only=False)

In [2]:
state.keys()

dict_keys(['model', 'optimizer', 'epoch', 'best_score', 'gradient_scaler', 'rng', 'lr_scheduler', 'args'])

In [9]:
state['model'].keys()

odict_keys(['temperature', 'bias', 'model.null_prompt', 'model.temperature', 'model.bias', 'model.model.image_encoder.backbone.cls_token', 'model.model.image_encoder.backbone.storage_tokens', 'model.model.image_encoder.backbone.mask_token', 'model.model.image_encoder.backbone.patch_embed.proj.weight', 'model.model.image_encoder.backbone.patch_embed.proj.bias', 'model.model.image_encoder.backbone.rope_embed.periods', 'model.model.image_encoder.backbone.blocks.0.norm1.weight', 'model.model.image_encoder.backbone.blocks.0.norm1.bias', 'model.model.image_encoder.backbone.blocks.0.attn.qkv.weight', 'model.model.image_encoder.backbone.blocks.0.attn.qkv.bias', 'model.model.image_encoder.backbone.blocks.0.attn.qkv.bias_mask', 'model.model.image_encoder.backbone.blocks.0.attn.proj.weight', 'model.model.image_encoder.backbone.blocks.0.attn.proj.bias', 'model.model.image_encoder.backbone.blocks.0.ls1.gamma', 'model.model.image_encoder.backbone.blocks.0.norm2.weight', 'model.model.image_encoder.ba

In [14]:
state['model']['model.no_mask_embed.weight'].size()

torch.Size([1, 512])

In [23]:
def dinov3_vitl16(**kwargs):
    import torch
    import os
    DINOV3_CHECKPOINTS_PATH='/datasets/exactvu_pca/checkpoint_store'
    REPO_DIR='/home/obed/projects/aip-medilab/obed/medproj/dinov3'

    weight_basename = "dinov3_vitl16_pretrain_lvd1689m-8aa4cbdd.pth"
    weights = os.path.join(DINOV3_CHECKPOINTS_PATH, weight_basename)
    if not os.path.exists(weights):
        raise FileNotFoundError(f"DINOv3 checkpoint not found: {weights}")
    kwargs["weights"] = weights
    model = torch.hub.load(REPO_DIR, "dinov3_vitl16", source="local", **kwargs)
    return model

In [7]:
from omegaconf import OmegaConf
from medAI.factories.prostnfound.models import get_model
from medAI.modeling.proside import Proside
from projects.dino_pca.train import ProstNFoundMeta

cfg = OmegaConf.load('/project/6106383/obed/medproj/medAI/projects/variational_primus/nct_cancer_l40s.yaml')

model = get_model(OmegaConf.to_object(cfg))
dino_model = Proside(model, **cfg.get('model_kw', {}))
pnf_model = ProstNFoundMeta(dino_model, **cfg.get("metamodel", {}))


# model = dinov3_vitl16()
# dino_model = Proside(model, **cfg.get('model_kw', {}))
# pnf_model = ProstNFoundMeta(dino_model, **cfg.get("metamodel", {}))

Model cfg: {'name': 'unetr_generic', 'backbone_cfg': {'name': 'dinov3_backbone_wrapper_for_feature_maps_list', 'backbone_cfg': {'name': 'dinov3_vitl16', 'pretrained': True}}, 'embedding_size': 1024, 'backbone_out_format': 'bchw', 'input_size': 256, 'output_size': 256, 'interaction_indices': [4, 11, 17, 23]}
Model: <class 'medAI.modeling.unetr.UNETR'>


In [8]:

model_state = state["model"]
pnf_model.load_state_dict(model_state, strict=False)

# dino_model.load_state_dict(state["model"])

<All keys matched successfully>